In [ ]:
from ib_insync import *
import asyncio

ib = IB()
await ib.connectAsync('127.0.0.1', 7497, clientId=1)  # paper

In [ ]:
symbols = ['IBIT','FBTC','ARKB','BITB','BTCW']

underlyings = []
for s in symbols:
    c = Stock(s, 'SMART', 'USD')
    await ib.qualifyContractsAsync(c)
    underlyings.append(c)

In [ ]:
chains = {}
for u in underlyings:
    params = await ib.reqSecDefOptParamsAsync(u.symbol, '', u.secType, u.conId)
    chains[u.symbol] = params[0]   # SMART chain

In [ ]:
options = []

for u in underlyings:
    chain = chains[u.symbol]
    spot = u.marketPrice()

    expiries = sorted(chain.expirations)[:3]
    strikes = [s for s in chain.strikes if abs(s-spot)/spot < 0.25]

    for exp in expiries:
        for strike in strikes:
            for right in ('C','P'):
                options.append(Option(u.symbol, exp, strike, right, 'SMART'))

await ib.qualifyContractsAsync(*options)


In [ ]:
ticks = []
for opt in options:
    t = ib.reqMktData(opt, snapshot=True)
    ticks.append(t)
    await asyncio.sleep(0.02)  # pacing protection

In [ ]:
await ib.sleep(2)

In [ ]:
import pandas as pd

rows = []
for t in ticks:
    c = t.contract
    rows.append({'symbol': c.symbol,
                 'expiry': c.lastTradeDateOrContractMonth,
                 'strike': c.strike,
                 'right': c.right,
                 'bid': t.bid,
                 'ask': t.ask,
                 'last': t.last,
                 'iv': t.impliedVolatility,
                 'delta': t.delta,
                 'gamma': t.gamma,
                 'theta': t.theta,
                 'vega': t.vega})

df = pd.DataFrame(rows)